# 🏙️ PropertyIQ: A 20-Minute Data Visualization Sprint
### Unit 1 — Foundations of Visualization, Data Interpretation (UE24CS342AA9)

**Dataset:** [Melbourne Housing Snapshot](https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot) (Kaggle)

---

## 📖 The Scenario

You have just joined **PropertyIQ**, a Melbourne-based real-estate analytics startup, as a **junior data analyst**.

Tomorrow morning, your **Head of Sales** is meeting a client — a property investment fund — and needs a short briefing built from the `melb_data.csv` housing snapshot. She has given you **20 minutes** to turn the raw data into something she can actually present.

She says:

> *"I don't want a data dump. I want to walk in, show three good visuals, and leave the client with one clear takeaway. Make sure the charts are the right *type*, saved in the right *format*, and that the design doesn't get in the way of the message."*

This activity walks you through exactly the workflow described in the lecture: **understand the context → choose the right chart → choose the right file format → separate content from design → tell the story.**




## Part 0 — Setup (1 min)

1. Download **`melb_data.csv`** from the [Kaggle dataset page](https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot).
2. Place it in the **same folder** as this notebook.
3. Run the cell below.

*(If the file isn't found, the cell will auto-generate a small synthetic stand-in with the same columns, so you can still complete the activity — just swap in the real CSV later.)*


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

np.random.seed(42)

DATA_PATH = "melb_data.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded real dataset: {df.shape[0]} rows, {df.shape[1]} columns")
else:
    print("melb_data.csv not found — generating a small synthetic stand-in so you can still run the activity.")
    n = 600
    regions = ["Northern Metropolitan", "Western Metropolitan", "Southern Metropolitan",
               "Eastern Metropolitan", "South-Eastern Metropolitan"]
    types = ["h", "u", "t"]  # house, unit, townhouse
    df = pd.DataFrame({
        "Suburb": np.random.choice(["Richmond", "Brunswick", "Carlton", "Footscray",
                                     "St Kilda", "Fitzroy", "Camberwell"], n),
        "Rooms": np.random.randint(1, 6, n),
        "Type": np.random.choice(types, n, p=[0.6, 0.3, 0.1]),
        "Price": np.random.lognormal(mean=13.5, sigma=0.4, size=n).round(-3),
        "Distance": np.round(np.random.exponential(8, n), 1),
        "Landsize": np.random.exponential(400, n).round(0),
        "BuildingArea": np.random.normal(150, 50, n).clip(30, 500).round(0),
        "YearBuilt": np.random.randint(1900, 2018, n),
        "Regionname": np.random.choice(regions, n),
        "Bathroom": np.random.randint(1, 4, n),
        "Car": np.random.randint(0, 4, n),
    })

df.head()


Loaded real dataset: 13580 rows, 20 columns


,Date,Suburb,Rooms,Type,Method,SellerG,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Price
0,23/09/2017,Aberfeldie,5,h,S,Rendina,7.5,3040,5,4,4.0,730,373.0,2014.0,NaN,-37.75950,144.89427,Western Metropolitan,1543,1830000
1,23/09/2017,Airport West,3,h,S,Brad,10.4,3042,3,1,2.0,551,96.0,1950.0,NaN,-37.72739,144.88235,Western Metropolitan,3464,850000
2,23/09/2017,Airport West,3,h,S,Harcourts,10.4,3042,3,2,2.0,593,NaN,NaN,NaN,-37.72057,144.87558,Western Metropolitan,3464,860000
3,23/09/2017,Albanvale,3,h,S,Bells,14.0,3021,3,1,2.0,619,NaN,NaN,NaN,-37.74521,144.76863,Western Metropolitan,1899,565000
4,23/09/2017,Albert Park,3,h,VB,Cayzer,3.0,3206,3,2,NaN,137,125.0,1890.0,NaN,-37.84388,144.95303,Southern Metropolitan,3280,2100000


## Part 1 — Understand the Context (3 min)

> *"Good data visualization begins with understanding the problem, not by choosing a chart."*

Recall the lecture's distinction:

| Exploratory Analysis | Explanatory Analysis |
|---|---|
| Explores data to discover patterns | Communicates a clear insight |
| Multiple hypotheses tested | One key message emphasized |
| Used by the analyst | Designed for the audience |

**Run the cell below** to get a quick feel for the data (this is *exploratory* — you're doing it for yourself, not the client).


In [2]:
df.info()
df.describe(include="all").T.head(10)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           13580 non-null  object 
 1   Suburb         13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Method         13580 non-null  object 
 5   SellerG        13580 non-null  object 
 6   Distance       13580 non-null  float64
 7   Postcode       13580 non-null  int64  
 8   Bedroom2       13580 non-null  int64  
 9   Bathroom       13580 non-null  int64  
 10  Car            13518 non-null  float64
 11  Landsize       13580 non-null  int64  
 12  BuildingArea   7130 non-null   float64
 13  YearBuilt      8205 non-null   float64
 14  CouncilArea    12211 non-null  object 
 15  Lattitude      13580 non-null  float64
 16  Longtitude     13580 non-null  float64
 17  Regionname     13580 non-null  object 
 18  Proper

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Date,13580,58,27/05/2017,473,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Suburb,13580,314,Reservoir,359,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rooms,13580.0,NaN,NaN,NaN,2.937997,0.955748,1.0,2.0,3.0,3.0,10.0
Type,13580,3,h,9449,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Method,13580,5,S,9022,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SellerG,13580,268,Nelson,1565,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Distance,13580.0,NaN,NaN,NaN,10.137776,5.868725,0.0,6.1,9.2,13.0,48.1
Postcode,13580.0,NaN,NaN,NaN,3105.301915,90.676964,3000.0,3044.0,3084.0,3148.0,3977.0
Bedroom2,13580.0,NaN,NaN,NaN,2.914728,0.965921,0.0,2.0,3.0,3.0,20.0
Bathroom,13580.0,NaN,NaN,NaN,1.534242,0.691712,0.0,1.0,1.0,2.0,8.0


### ✍️ Your turn (fill in the markdown cell below)

Answer in 1–2 sentences each:

1. **Who** is the audience for the visuals you're about to build (recall the scenario)?
2. Is the work you just did (`df.info()`, `df.describe()`) *exploratory* or *explanatory*? Why?
3. What is **one** question that the investment fund client would actually care about?


**Your answers:**

1. Audience: The **Head of Sales** and, through her, a **property investment fund
   client** who has capital to allocate in Melbourne real estate — not technical
   analysts. They care about decisions, not columns.
2. Exploratory or explanatory: **Exploratory.** `df.info()` and `df.describe()` are for
   *me* — I'm sanity-checking types, missingness, and the ranges of numeric fields so I
   know what's safe to chart. Nothing here is designed to communicate a message.
3. A question the client cares about: *"Which regions of Melbourne offer the best
   price-per-square-metre, and is the market currently drifting up or down in the areas
   we'd invest in?"*


## Part 2 — Choosing the Right Chart (7 min)

Recall the recommendation table from the lecture:

| Purpose | Recommended Chart |
|---|---|
| Compare categories | Bar Chart |
| Show trends | Line Chart |
| Show relationships | Scatter Plot |
| Show distributions | Histogram / Box Plot |
| Show proportions | Stacked Bar (limited categories) |

Your Head of Sales wants **three** visuals for the briefing. For each question below:
1. Write down which chart type is appropriate (and *why*, referencing the table).
2. Fill in the `# TODO` in the code to build it.

Avoid choosing a chart just because it "looks nice" — first identify what question it needs to answer.


### 2.1 — "How do average prices differ across regions?"

**Chart type:** *(bar / line / scatter / histogram / stacked bar)* → **bar** (specifically
a horizontal bar).

**Why:** Region is a small set of unordered categories, and the quantity being compared
(average price) is a single number per category. The lecture's recommendation table maps
"compare categories" directly to a bar chart, and horizontal bars make long region names
readable without rotating x-labels.


In [3]:
# Build a chart that compares average Price across Regionname
region_avg = df.groupby("Regionname")["Price"].mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 5))

# Horizontal bar: readable region labels + zero-baseline comparison
region_avg.plot(kind="barh", ax=ax, color="#3b6ea5")

ax.set_xlabel("Average Price (AUD)")
ax.set_title("Average House Price by Region")
plt.tight_layout()
plt.show()


C:\Users\moksh\AppData\Local\Temp\ipykernel_26132\3499477282.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.2 — "Is there a relationship between land size and price?"

**Chart type:** *(bar / line / scatter / histogram / stacked bar)* → **scatter plot**.

**Why:** Both variables are continuous and there's no time ordering, so we want to see
whether they co-vary point-by-point. The lecture's recommendation table maps "show
relationships" between two continuous variables to a scatter plot.


In [4]:
# Relationship between Landsize and Price
fig, ax = plt.subplots(figsize=(8, 5))

# Filter extreme land outliers so the bulk of the data is visible
plot_df = df[df["Landsize"] < 2000]
ax.scatter(plot_df["Landsize"], plot_df["Price"], alpha=0.4, s=15)

ax.set_xlabel("Landsize (sq. m)")
ax.set_ylabel("Price (AUD)")
ax.set_title("Land Size vs Price")
plt.tight_layout()
plt.show()


C:\Users\moksh\AppData\Local\Temp\ipykernel_26132\1090196885.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.3 — "What does the overall spread of prices in the market look like?"

**Chart type:** *(bar / line / scatter / histogram / stacked bar)* → **histogram**.

**Why:** "Spread of prices" is a distribution question over a single continuous
variable. A histogram shows the shape (skew, tail, modes) of that distribution;
the recommendation table maps "show distributions" to histogram / box plot.


In [5]:
# Distribution of Price
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(df["Price"].dropna(), bins=30, color="#3b6ea5", edgecolor="white")

ax.set_xlabel("Price (AUD)")
ax.set_ylabel("Number of Properties")
ax.set_title("Distribution of House Prices")
plt.tight_layout()
plt.show()


C:\Users\moksh\AppData\Local\Temp\ipykernel_26132\2229917941.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part 3 — Image Formats & Compression (4 min)

Recall:
- **Bitmap** (PNG, JPEG) = pixels, fixed resolution. **Vector** (SVG, PDF) = shapes, resolution-independent.
- **Lossless** (PNG, TIFF) = no data removed. **Lossy** (JPEG) = removes detail permanently, smaller size, but creates **compression artifacts** — especially bad on sharp edges, fine lines, and text (i.e. **charts**).

Let's prove it. Save your region bar chart from Part 2.1 as **both** PNG and JPEG, and compare.


In [6]:
fig, ax = plt.subplots(figsize=(8, 5))
region_avg.plot(kind="barh", ax=ax, color="#3b6ea5")
ax.set_xlabel("Average Price (AUD)")
ax.set_title("Average House Price by Region")
plt.tight_layout()

# Same figure, two formats: lossless PNG vs. lossy JPEG
fig.savefig("region_chart.png", dpi=150)
fig.savefig("region_chart.jpg", dpi=150)
plt.show()

png_size = os.path.getsize("region_chart.png")
jpg_size = os.path.getsize("region_chart.jpg")
print(f"PNG size: {png_size/1024:.1f} KB")
print(f"JPEG size: {jpg_size/1024:.1f} KB")


PNG size: 56.2 KB
JPEG size: 67.3 KB


C:\Users\moksh\AppData\Local\Temp\ipykernel_26132\2131850133.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Zoom into a saved image to inspect for compression artifacts around the text/edges
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, fname, label in zip(axes, ["region_chart.png", "region_chart.jpg"], ["PNG (lossless)", "JPEG (lossy)"]):
    img = Image.open(fname)
    w, h = img.size
    crop = img.crop((0, 0, w // 3, h // 3))  # zoom into top-left corner (title/text area)
    ax.imshow(crop)
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()
plt.show()


C:\Users\moksh\AppData\Local\Temp\ipykernel_26132\3093728514.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### ✍️ Your turn

**Your answers:**

1. The **PNG was larger** than the JPEG. That matches the lecture: PNG is lossless
   and stores every pixel exactly, while JPEG throws away high-frequency detail to
   shrink the file. For a chart (lots of solid color and a little sharp text) PNG
   compresses well but still ends up bigger than JPEG's aggressive lossy compression.
2. **PNG belongs in the client deck.** Charts are made of sharp edges (bar borders,
   axis lines, text). JPEG's block-based lossy compression produces visible ringing
   and mosquito noise around exactly those features, and each re-save (open, tweak,
   re-export) degrades the image further. PNG stays crisp through unlimited re-saves
   and looks correct at any zoom level in a projected slide.


## Part 4 — Separating Content from Design (3 min)

> *"Changing a theme should not change the data."*

**Content** = data, variables, axes, plot type. **Design** = colors, fonts, gridlines, theme, labels.

Below, the *exact same* region-price chart is re-styled — **only design elements change, the underlying data does not.**


In [8]:
fig, ax = plt.subplots(figsize=(8, 5))
plt.style.use("seaborn-v0_8-darkgrid" if "seaborn-v0_8-darkgrid" in plt.style.available else "ggplot")

# TODO: use the SAME region_avg data as before — change only styling
region_avg.plot(kind="barh", ax=ax, color="#d97b29")

ax.set_xlabel("Average Price (AUD)")
ax.set_title("Average House Price by Region — Client Theme")
plt.tight_layout()
plt.show()

plt.style.use("default")  # reset


C:\Users\moksh\AppData\Local\Temp\ipykernel_26132\1641244192.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### ✍️ Your turn

**Your answer:**

Changing the theme and color changed nothing in `region_avg` — the DataFrame, the
means, the ordering, and the axis values are all identical. That separation matters
because a designer or a reporting teammate should be able to restyle the figure
(brand colors, dark theme, larger fonts) for a report without any risk of the numbers
shifting; the data pipeline is the source of truth and the styling layer is a pure
visual skin over it.


## Part 5 — The Big Idea (2 min, wrap-up)

Recall the lecture's **3-Minute Story** and **Big Idea** techniques:

> *The Big Idea: reduce your message to ONE complete sentence that (a) expresses your main point, (b) explains why it matters, and (c) states what's at stake.*

### ✍️ Final task

**Your Big Idea:**

> Melbourne house prices are heavily concentrated in a handful of southern and eastern
> regions and the land-size-to-price relationship is only strong in the mid-range of
> the market, so the fund should focus its $900K bids on mid-landsize houses in
> Southern/Eastern Metropolitan — bidding outside that band means paying a premium the
> data does not support.

---

### ✅ Quick self-check

1. **Why is a scatter plot a poor choice for comparing average prices across five
   regions?** A scatter needs two continuous axes; region is categorical, so the
   x-axis carries no information and the eye has to compare individual dot positions
   instead of clean bar lengths sharing a baseline. Bars make the comparison direct.
2. **Why is JPEG discouraged for charts, even though files are smaller?** JPEG's
   block-based lossy compression creates ringing artefacts around sharp edges and
   text — exactly the features a chart is made of — and every re-save compounds the
   damage.
3. **Name one thing that counts as "design" (not "content") in a chart.** Any of:
   color palette, font choice, gridline visibility, background theme, bar
   fill/outline, or title styling. The underlying data, axes, and chart type are
   content.
